# 0 基础版生成（无检索 无个性化）

In [ ]:
import os
from dotenv import load_dotenv
from langchain_community.graphs import Neo4jGraph
import os
from langchain_openai import ChatOpenAI
load_dotenv()
os.environ["DEEPSEEK_API_KEY"] = os.getenv("DEEPSEEK_API_KEY")
os.environ["QWEN_API_KEY"] = os.getenv("QWEN_API_KEY")



In [ ]:
# 1.读取neo4j数据库
load_dotenv()
url = os.getenv('NEO4J_URL')
username = os.getenv('NEO4J_USER')
password = os.getenv('NEO4J_PASSWORD')

C:\Users\lenovo\AppData\Local\Temp\ipykernel_32840\3089890364.py:3: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(


In [51]:
# 2.定义LLM
from langchain.chat_models import init_chat_model

llm_ds=init_chat_model(
    model='deepseek:deepseek-chat',
    temperature=1.3)

llm_qwen=ChatOpenAI(
    model="qwen-max",
    openai_api_key=os.getenv("QWEN_API_KEY"),
    openai_api_base="https://dashscope.aliyuncs.com/compatible-mode/v1",
    temperature=1.3
)

In [4]:
# 3.图检索
#基于cypher
#返回子图
def retrieve_1hop_subgraph(graph, keyword):
    query = """
    MATCH (n {id: $kw})
    OPTIONAL MATCH (n)-[r]-(m)
    RETURN 
        COLLECT(DISTINCT n) + COLLECT(DISTINCT m) AS nodes,
        COLLECT(DISTINCT r) AS relationships
    """
    res = graph.query(query, {"kw": keyword})
    if not res or not res[0]["nodes"]:
        return {"nodes": [], "relationships": []}
    return {
        "nodes": res[0]["nodes"],
        "relationships": res[0]["relationships"]
    }

In [5]:
# 4.从图构建上下文
# 检索出的子图数据（即相关知识）-->文本，作为生成的知识上下文
def get_knowledge_context(subgraph):
    sentences = set()
    
    for rel in subgraph["relationships"]:
        source = rel[0]['id']
        target = rel[2]['id']
        rtype = rel[1]
        # 转换为自然语言句子
        sentence = f"{source}{rtype}{target}。"
        sentences.add(sentence)
    
    return "\n".join(sorted(sentences))

In [6]:
#测试
keyword='冰川地貌'
retrieved_results=retrieve_1hop_subgraph(graph, keyword)
print('检索出子图：\n',retrieved_results)

context=get_knowledge_context(retrieved_results)
print('\n检索出背景知识：\n',context)

检索出子图：
 {'nodes': [{'id': '冰川地貌'}, {'id': '冰川槽谷'}, {'id': '峡湾'}, {'id': '地貌'}, {'id': '角峰'}, {'id': '刃脊'}, {'id': '冰斗'}, {'id': '挪威西峡湾'}, {'id': '冰川'}], 'relationships': [({'id': '冰川地貌'}, '包含', {'id': '冰川槽谷'}), ({'id': '冰川地貌'}, '包含', {'id': '峡湾'}), ({'id': '地貌'}, '包含', {'id': '冰川地貌'}), ({'id': '冰川地貌'}, '包含', {'id': '角峰'}), ({'id': '冰川地貌'}, '包含', {'id': '刃脊'}), ({'id': '冰川地貌'}, '包含', {'id': '冰斗'}), ({'id': '冰川地貌'}, '形成于', {'id': '挪威西峡湾'}), ({'id': '冰川'}, '塑造', {'id': '冰川地貌'})]}

检索出背景知识：
 冰川地貌包含冰川槽谷。
冰川地貌包含冰斗。
冰川地貌包含刃脊。
冰川地貌包含峡湾。
冰川地貌包含角峰。
冰川地貌形成于挪威西峡湾。
冰川塑造冰川地貌。
地貌包含冰川地貌。


In [6]:
# 5.生成
from langchain_core.prompts import PromptTemplate
answer_propmt=PromptTemplate.from_template(
    '''
    你是一名高中地理资深命题教师。
    
    【已知条件】
    - 知识上下文：
    {context}
    - 目标知识点：{knowledge_point}
    - 难度等级：{difficulty}
    - 题型：{q_type}
    - 是否涉及运算：{calculation}
    - 认知层级（如：记忆 / 理解 / 应用 / 分析 / 评价 / 创造）：
    {cognitive_level}
    
    【出题要求】
    1. 题目必须以「目标知识点」为核心  
    2. 可合理使用上下文中的相关知识点作为支撑，但只能起辅助作用  
    3. 严格匹配给定的题型、难度、是否运算、认知层级
    4. 题干表述清晰，不引入无关信息。纯文本题，不要用到图片。
    5. 设问方式灵活，所问的可能是下一句或者选择一个陈述句（这两种情况最多），也可能是挖空，也可能是从下列选择或排序等
    5. 有概率用真实地理景观为例出题，但不要只用上下文中的，可以用其他的，比如“乞拉朋齐……据此请分析……的成因主要是”
    5. 只生成一道题，不要解释、不输出思路
    6. 要符合高中地理考题风格

    严格按照以下格式（包括换行也要一致）输出，不要含有其他内容如前后缀等，最前面不要有空格
    【输出格式】：
    【{difficulty}题 | {cognitive_level}题 | 运算题（如果涉及运算的话）】前两个xx分别填难度、认知层级
    【题目】
    （另起一行）题目内容
    （如是选择题）【选项】
    【参考答案】
    （另起一行）（如果是选择题，只输出选项一个字母）
    '''
)
                         

In [7]:
# 完整生成pipeline
def rag(knowledge_point,difficulty,
        q_type,calculation,cognitive_level):
    # 检索出的子图
    graph_results=retrieve_1hop_subgraph(graph,knowledge_point)
    # 子图->知识上下文
    context=get_knowledge_context(graph_results)
    response=llm.invoke(
        answer_propmt.format(
            context=context,
            knowledge_point=knowledge_point,
            difficulty=difficulty,
            q_type=q_type,
            calculation=calculation,
            cognitive_level=cognitive_level
        )
    )

    return graph_results,context,response.content

In [9]:
# 6.运行
difficulty_list=['简单','中等','困难']
q_type_list=['选择题']
calculation_list=['不涉及','涉及']
cognitive_levels=['记忆','理解','应用','分析','评价','创造']

# 1 引入题库特征+学生背景建模
以上是手动指定试题生成要求

以下引入题库特征+学生背景建模

In [8]:
# 试题库特征建模，存成字典
# 以下的block表示小节
block2id={'流水地貌':0,'风成地貌':1,'喀斯特、海岸和冰川地貌':2}
feature_map={
    '流水地貌':{
        'k_points':["地貌", "流水地貌", "流水侵蚀地貌", "流水堆积地貌", "峡谷", "河漫滩", "河流阶地", "曲流", "牛轭湖", "冲积扇", "洪积扇", "冲积平原", "三角洲", "江心洲", "滑坡", "泥石流", "流水", "山洪", "虎跳峡", "雅鲁藏布大峡谷", "嘉陵江", "崇明岛", "橘子洲", "黄河三角洲", "尼罗河三角洲", "华北平原", "东北平原", "长江中下游平原"],
        'difficulty':[33/43,8/43,2/43],
        'q_type':[1], 
        'calculation':0.0,
        'cognitive_levels':[5/43,30/43,0/43,8/43,0/43,0/43]
    },
    '风成地貌':{
        'k_points':["风蚀地貌", "风积地貌", "风蚀蘑菇", "风蚀壁龛", "风蚀柱", "雅丹地貌", "沙丘", "新月形沙丘", "灌丛沙丘", "风力"],
        'difficulty':[13/23,10/23,0/23],
        'q_type':[1], 
        'calculation':0.0,
        'cognitive_levels':[12/23,7/23,1/23,3/23,0/23,0/23]
    },
    '喀斯特、海岸和冰川地貌':{
        'k_points':["喀斯特地貌", "海岸地貌", "冰川地貌", "海蚀地貌", "海积地貌", "海蚀崖", "海蚀平台", "海蚀柱", "海滩", "岬角", "冰斗", "冰川槽谷", "角峰", "刃脊", "峡湾", "波浪", "冰川", "云南石林", "广西桂林", "重庆奉节小寨天坑", "贵州荔波", "四川黄龙", "湖南张家界黄龙洞", "澳大利亚坎贝尔港国家公园", "挪威西峡湾"],
        'difficulty':[17/37,1/37,19/37],
        'q_type':[1], 
        'calculation':0.0,
        'cognitive_levels':[15/37,4/37,2/37,16/37,0/37,0/37]
    }
}

In [9]:
# 学生背景，存成字典
stu_map=[
    {'handle':0.5,
     'calculation':0.7,
     'cognitive_levels':3
    },
    {'handle':0.2,
     'calculation':0.5,
     'cognitive_levels':2
    },
    {'handle':0.7,
     'calculation':0.5,
     'cognitive_levels':4
    }
     ]

In [ ]:
# 试题特征融合
from math import exp

def merge_feature(block,stu_group):
    # 0.取出当前学生的背景、当前小节的题库分布
    # 假设细胞外液知识点 属于 生理指标小节
    # block_id=block2id[block]
    true_feature=feature_map[block]
    stu_bg=stu_map[stu_group]

    # 1.难度
    t=0.5 #温度系数
    difficulty_spot=[0.2,0.5,0.85] #三个难度对应的标准难度系数
    difficulty=true_feature['difficulty']
    difficulty_p=[] #最后的难度概率分布
    sum_p=0
    for i in range(3):
        distance=abs(difficulty_spot[i]-stu_bg['handle'])
        w_i=exp(-distance/t)
        p=difficulty[i]*w_i
        difficulty_p.append(p)
        sum_p+=p
    for i in range(3):  #归一化
        difficulty_p[i]=difficulty_p[i]/sum_p
        
    # 2.涉及运算概率
    # 1.5的来历是设运算能力=0.5为中等值，这样当运算能力>0.5，运算题概率会被缩小
    calculation=true_feature['calculation']*(1.5-stu_bg['calculation'])
    
    # 3.认知层级
    decay1=[0.3,0.4,0.5,0.7,0.8,1]
    decay2=[1,0.5,0.25,0.1,0.1,0.1]
    cognitive_levels=true_feature['cognitive_levels']
    c=stu_bg['cognitive_levels']
    cognitive_p=[0 for _ in range(6)]
    for i in range(c+1):
        cognitive_p[i]=cognitive_levels[i]*decay1[6-c+i-1]
    if c<5:
        for i in range(c+1,6):
            cognitive_p[i]=cognitive_levels[i]*decay2[i-c]
    sum_p=sum(cognitive_p)
    for i in range(6):
        cognitive_p[i]=cognitive_p[i]/sum_p

    # print(true_feature)
    # print(stu_bg)
    # print(difficulty_p)
    # print(calculation)
    # print(cognitive_p)

    return difficulty_p,calculation,cognitive_p

In [11]:
# 采样 获取单次生成的具体取值
import random
difficulty_list=['简单','中等','困难']
q_type_list=['选择题']
calculation_list=['不涉及','涉及']
cognitive_levels=['记忆','理解','应用','分析','评价','创造']

def choose_feature(difficulty_p,calculation,cognitive_p):
    # 1. 抽难度（0:简单 1:中等 2:困难）
    cur_difficulty = random.choices([0, 1, 2], weights=difficulty_p, k=1)[0]
    cur_difficulty=difficulty_list[cur_difficulty]
    # 2. 抽是否运算（0/1）
    cur_calculation = random.random() < calculation
    cur_calculation=calculation_list[cur_calculation]
    # 3. 抽认知层级（0~5）
    cur_cognitive_level = random.choices(range(6), weights=cognitive_p, k=1)[0]
    cur_cognitive_level=cognitive_levels[cur_cognitive_level]
    
    print(cur_difficulty,cur_calculation,cur_cognitive_level)
    return cur_difficulty,cur_calculation,cur_cognitive_level

In [ ]:
# 生成
# knowledge_point可以不指定（只指定小节block），也可以指定
def get_questions(block,stu_group,knowledge_point=None):
    # 抽取生成要求
    difficulty_p,calculation,cognitive_p=merge_feature(
        block=block,
        stu_group=stu_group)
    
    cur_difficulty,cur_calculation,cur_cognitive_level=choose_feature(
        difficulty_p,calculation,cognitive_p
    )
    # 抽取知识点
    all_kps=feature_map[block]['k_points']
    if not knowledge_point or knowledge_point not in allkps:
        knowledge_point=random.choices(all_kps,k=1)[0]
    # 生成
    graph_results,context,answer=rag(
        knowledge_point=knowledge_point,
        difficulty=cur_difficulty,
        q_type=q_type_list[0],
        calculation=cur_calculation,
        cognitive_level=cur_cognitive_level
    )

    features=[knowledge_point,cur_difficulty,q_type_list[0],
              cur_calculation,cur_cognitive_level,]

    return answer,features,context

# print('graph_results:\n',graph_results)
# print('context:\n',context)
# print('A:\n',answer)

In [13]:
# 解析llm的返回->结构化
def parse(answer:str, idx:int, features,context):
    # print(answer)
    lines = [l.strip() for l in answer.strip().splitlines() if l.strip()]
    # print(lines)
    # 题干、选项、参考答案
    q_start = lines.index("【题目】") + 1
    o_start = lines.index("【选项】")
    a_start = lines.index("【参考答案】") + 1

    question = " ".join(lines[q_start:o_start])
    options = " ".join(lines[o_start + 1:a_start - 1])
    answer_text = lines[a_start]

    context = context.replace('\n', '')

    features.append(question)
    features.append(options)
    features.append(answer_text)
    features.append(context)
    return features

# 2 批量GENERATE

In [16]:
llm = llm_ds
n = 500
save_path = './output/ds-210-instruction-data.txt'

count = 0  # 成功生成的题目数
attempt = 0  # 总尝试次数（防死循环）

with open(save_path, "w", encoding="utf-8") as f:
    while count < n and attempt < n * 3:  # 最多尝试 3n 次
        attempt += 1
        print(f'正在生成第{count+1}题（尝试第{attempt}次）：', end=' ')
        try:
            answer, features, context = get_questions(
                block='流水地貌',
                stu_group=0,
                knowledge_point=None
            )
            features = parse(answer, count, features, context)  # 这里用 count 做题号
            f.write(";".join(features) + "\n")
            f.flush()  # 立即写入，避免丢失
            count += 1
            print('成功')
        except Exception as e:
            print(f'【生成有误，重试】错误信息：{e}')

    print(f'生成完成，共成功 {count} 题，保存至 {save_path}')

正在生成第1题（尝试第1次）： 简单 不涉及 分析
成功
正在生成第2题（尝试第2次）： 简单 不涉及 分析
成功
正在生成第3题（尝试第3次）： 简单 不涉及 理解
成功
正在生成第4题（尝试第4次）： 简单 不涉及 理解
成功
正在生成第5题（尝试第5次）： 简单 不涉及 分析
成功
正在生成第6题（尝试第6次）： 简单 不涉及 理解
成功
正在生成第7题（尝试第7次）： 简单 不涉及 理解
成功
正在生成第8题（尝试第8次）： 中等 不涉及 理解
成功
正在生成第9题（尝试第9次）： 简单 不涉及 理解
成功
正在生成第10题（尝试第10次）： 中等 不涉及 理解
成功
正在生成第11题（尝试第11次）： 简单 不涉及 理解
成功
正在生成第12题（尝试第12次）： 简单 不涉及 理解
成功
正在生成第13题（尝试第13次）： 简单 不涉及 分析
成功
正在生成第14题（尝试第14次）： 中等 不涉及 理解
成功
正在生成第15题（尝试第15次）： 中等 不涉及 理解
成功
正在生成第16题（尝试第16次）： 中等 不涉及 理解
成功
正在生成第17题（尝试第17次）： 中等 不涉及 理解
成功
正在生成第18题（尝试第18次）： 简单 不涉及 分析
成功
正在生成第19题（尝试第19次）： 简单 不涉及 理解
成功
正在生成第20题（尝试第20次）： 简单 不涉及 分析
成功
正在生成第21题（尝试第21次）： 简单 不涉及 理解
成功
正在生成第22题（尝试第22次）： 中等 不涉及 理解
成功
正在生成第23题（尝试第23次）： 简单 不涉及 理解
成功
正在生成第24题（尝试第24次）： 简单 不涉及 理解
成功
正在生成第25题（尝试第25次）： 简单 不涉及 理解
成功
正在生成第26题（尝试第26次）： 简单 不涉及 理解
成功
正在生成第27题（尝试第27次）： 简单 不涉及 理解
成功
正在生成第28题（尝试第28次）： 中等 不涉及 分析
成功
正在生成第29题（尝试第29次）： 中等 不涉及 分析
成功
正在生成第30题（尝试第30次）： 简单 不涉及 理解
成功
正在生成第31题（尝试第31次）： 简单 不涉及 理解
成功
正在生成第32题（尝试第32次）： 中等 不涉及 理解
成功
正在生成第33题（尝试第33次）： 中等 不涉及 理

In [17]:
llm = llm_ds
n = 500
save_path = './output/ds-220-instruction-data.txt'

count = 0  # 成功生成的题目数
attempt = 0  # 总尝试次数（防死循环）

with open(save_path, "w", encoding="utf-8") as f:
    while count < n and attempt < n * 3:  # 最多尝试 3n 次
        attempt += 1
        print(f'正在生成第{count+1}题（尝试第{attempt}次）：', end=' ')
        try:
            answer, features, context = get_questions(
                block='风成地貌',
                stu_group=0,
                knowledge_point=None
            )
            features = parse(answer, count, features, context)  # 这里用 count 做题号
            f.write(";".join(features) + "\n")
            f.flush()  # 立即写入，避免丢失
            count += 1
            print('成功')
        except Exception as e:
            print(f'【生成有误，重试】错误信息：{e}')

    print(f'生成完成，共成功 {count} 题，保存至 {save_path}')

正在生成第1题（尝试第1次）： 中等 不涉及 理解
成功
正在生成第2题（尝试第2次）： 中等 不涉及 理解
成功
正在生成第3题（尝试第3次）： 中等 不涉及 理解
成功
正在生成第4题（尝试第4次）： 简单 不涉及 理解
成功
正在生成第5题（尝试第5次）： 简单 不涉及 记忆
成功
正在生成第6题（尝试第6次）： 简单 不涉及 理解
成功
正在生成第7题（尝试第7次）： 简单 不涉及 记忆
成功
正在生成第8题（尝试第8次）： 简单 不涉及 分析
成功
正在生成第9题（尝试第9次）： 中等 不涉及 应用
成功
正在生成第10题（尝试第10次）： 简单 不涉及 理解
成功
正在生成第11题（尝试第11次）： 中等 不涉及 分析
成功
正在生成第12题（尝试第12次）： 中等 不涉及 理解
成功
正在生成第13题（尝试第13次）： 中等 不涉及 记忆
成功
正在生成第14题（尝试第14次）： 简单 不涉及 分析
成功
正在生成第15题（尝试第15次）： 中等 不涉及 理解
成功
正在生成第16题（尝试第16次）： 中等 不涉及 分析
成功
正在生成第17题（尝试第17次）： 中等 不涉及 记忆
成功
正在生成第18题（尝试第18次）： 简单 不涉及 理解
成功
正在生成第19题（尝试第19次）： 简单 不涉及 记忆
成功
正在生成第20题（尝试第20次）： 中等 不涉及 理解
成功
正在生成第21题（尝试第21次）： 中等 不涉及 记忆
成功
正在生成第22题（尝试第22次）： 中等 不涉及 理解
成功
正在生成第23题（尝试第23次）： 中等 不涉及 分析
成功
正在生成第24题（尝试第24次）： 中等 不涉及 记忆
成功
正在生成第25题（尝试第25次）： 中等 不涉及 记忆
成功
正在生成第26题（尝试第26次）： 中等 不涉及 记忆
成功
正在生成第27题（尝试第27次）： 中等 不涉及 应用
成功
正在生成第28题（尝试第28次）： 中等 不涉及 理解
成功
正在生成第29题（尝试第29次）： 简单 不涉及 分析
成功
正在生成第30题（尝试第30次）： 中等 不涉及 分析
成功
正在生成第31题（尝试第31次）： 中等 不涉及 分析
成功
正在生成第32题（尝试第32次）： 简单 不涉及 分析
成功
正在生成第33题（尝试第33次）： 简单 不涉及 记

In [18]:
llm = llm_qwen
n = 500
save_path = './output/qwen-210-instruction-data.txt'

count = 0  # 成功生成的题目数
attempt = 0  # 总尝试次数（防死循环）

with open(save_path, "w", encoding="utf-8") as f:
    while count < n and attempt < n * 3:  # 最多尝试 3n 次
        attempt += 1
        print(f'正在生成第{count+1}题（尝试第{attempt}次）：', end=' ')
        try:
            answer, features, context = get_questions(
                block='流水地貌',
                stu_group=0,
                knowledge_point=None
            )
            features = parse(answer, count, features, context)  # 这里用 count 做题号
            f.write(";".join(features) + "\n")
            f.flush()  # 立即写入，避免丢失
            count += 1
            print('成功')
        except Exception as e:
            print(f'【生成有误，重试】错误信息：{e}')

    print(f'生成完成，共成功 {count} 题，保存至 {save_path}')

正在生成第1题（尝试第1次）： 中等 不涉及 分析
【生成有误，重试】错误信息：'【题目】' is not in list
正在生成第1题（尝试第2次）： 中等 不涉及 理解
成功
正在生成第2题（尝试第3次）： 简单 不涉及 记忆
成功
正在生成第3题（尝试第4次）： 中等 不涉及 理解
成功
正在生成第4题（尝试第5次）： 简单 不涉及 理解
成功
正在生成第5题（尝试第6次）： 简单 不涉及 记忆
成功
正在生成第6题（尝试第7次）： 中等 不涉及 理解
成功
正在生成第7题（尝试第8次）： 简单 不涉及 记忆
成功
正在生成第8题（尝试第9次）： 中等 不涉及 分析
【生成有误，重试】错误信息：'【题目】' is not in list
正在生成第8题（尝试第10次）： 中等 不涉及 分析
成功
正在生成第9题（尝试第11次）： 简单 不涉及 理解
成功
正在生成第10题（尝试第12次）： 简单 不涉及 理解
成功
正在生成第11题（尝试第13次）： 简单 不涉及 理解
成功
正在生成第12题（尝试第14次）： 简单 不涉及 分析
成功
正在生成第13题（尝试第15次）： 简单 不涉及 理解
成功
正在生成第14题（尝试第16次）： 中等 不涉及 理解
成功
正在生成第15题（尝试第17次）： 简单 不涉及 理解
成功
正在生成第16题（尝试第18次）： 中等 不涉及 理解
成功
正在生成第17题（尝试第19次）： 困难 不涉及 理解
成功
正在生成第18题（尝试第20次）： 简单 不涉及 分析
成功
正在生成第19题（尝试第21次）： 简单 不涉及 分析
成功
正在生成第20题（尝试第22次）： 简单 不涉及 理解
成功
正在生成第21题（尝试第23次）： 简单 不涉及 记忆
成功
正在生成第22题（尝试第24次）： 简单 不涉及 理解
成功
正在生成第23题（尝试第25次）： 简单 不涉及 理解
成功
正在生成第24题（尝试第26次）： 简单 不涉及 理解
成功
正在生成第25题（尝试第27次）： 简单 不涉及 分析
成功
正在生成第26题（尝试第28次）： 中等 不涉及 理解
成功
正在生成第27题（尝试第29次）： 简单 不涉及 理解
成功
正在生成第28题（尝试第30次）： 简单 不涉及 理解
成功
正在生成第29题（尝试第31次）： 简单 不涉及

In [19]:
llm = llm_qwen
n = 500
save_path = './output/qwen-220-instruction-data.txt'

count = 0  # 成功生成的题目数
attempt = 0  # 总尝试次数（防死循环）

with open(save_path, "w", encoding="utf-8") as f:
    while count < n and attempt < n * 3:  # 最多尝试 3n 次
        attempt += 1
        print(f'正在生成第{count+1}题（尝试第{attempt}次）：', end=' ')
        try:
            answer, features, context = get_questions(
                block='风成地貌',
                stu_group=0,
                knowledge_point=None
            )
            features = parse(answer, count, features, context)  # 这里用 count 做题号
            f.write(";".join(features) + "\n")
            f.flush()  # 立即写入，避免丢失
            count += 1
            print('成功')
        except Exception as e:
            print(f'【生成有误，重试】错误信息：{e}')

    print(f'生成完成，共成功 {count} 题，保存至 {save_path}')

正在生成第1题（尝试第1次）： 中等 不涉及 理解
成功
正在生成第2题（尝试第2次）： 简单 不涉及 记忆
成功
正在生成第3题（尝试第3次）： 中等 不涉及 理解
成功
正在生成第4题（尝试第4次）： 中等 不涉及 理解
成功
正在生成第5题（尝试第5次）： 简单 不涉及 记忆
成功
正在生成第6题（尝试第6次）： 中等 不涉及 理解
成功
正在生成第7题（尝试第7次）： 中等 不涉及 理解
成功
正在生成第8题（尝试第8次）： 简单 不涉及 分析
成功
正在生成第9题（尝试第9次）： 中等 不涉及 分析
成功
正在生成第10题（尝试第10次）： 中等 不涉及 分析
成功
正在生成第11题（尝试第11次）： 简单 不涉及 应用
成功
正在生成第12题（尝试第12次）： 中等 不涉及 理解
成功
正在生成第13题（尝试第13次）： 简单 不涉及 分析
成功
正在生成第14题（尝试第14次）： 简单 不涉及 记忆
成功
正在生成第15题（尝试第15次）： 简单 不涉及 理解
成功
正在生成第16题（尝试第16次）： 中等 不涉及 记忆
成功
正在生成第17题（尝试第17次）： 简单 不涉及 理解
成功
正在生成第18题（尝试第18次）： 简单 不涉及 理解
成功
正在生成第19题（尝试第19次）： 中等 不涉及 分析
成功
正在生成第20题（尝试第20次）： 简单 不涉及 理解
成功
正在生成第21题（尝试第21次）： 简单 不涉及 理解
成功
正在生成第22题（尝试第22次）： 中等 不涉及 记忆
成功
正在生成第23题（尝试第23次）： 中等 不涉及 记忆
成功
正在生成第24题（尝试第24次）： 中等 不涉及 记忆
成功
正在生成第25题（尝试第25次）： 中等 不涉及 应用
成功
正在生成第26题（尝试第26次）： 简单 不涉及 应用
成功
正在生成第27题（尝试第27次）： 中等 不涉及 记忆
成功
正在生成第28题（尝试第28次）： 中等 不涉及 记忆
成功
正在生成第29题（尝试第29次）： 简单 不涉及 理解
成功
正在生成第30题（尝试第30次）： 简单 不涉及 记忆
成功
正在生成第31题（尝试第31次）： 简单 不涉及 理解
成功
正在生成第32题（尝试第32次）： 中等 不涉及 分析
成功
正在生成第33题（尝试第33次）： 简单 不涉及 理

# 3 对照组

## 3.1 无检索
A组：知识点->LLM生成  
B组：知识点->检索->prompt注入子图语段->LLM  
评估：  
1）知识相关性：对相同知识点的A/B组试题，比较输出试题 与 子图上下文 的topk子句语义相似度，A组的参照上下文就用该知识点对应的检索结果  
2）可解释性：无变化，均采用多次作答评估即可  
3）难度准确性：无变化，均采用同{难度、章节}题库试题 语义相似度 即可

In [14]:
# A组 直接LLM生成
#提示词模板
aprompt_noncontext=PromptTemplate.from_template(
    '''
    你是一名高中地理资深命题教师，请根据以下任务说明进行出题。
    
    【已知条件】
    - 目标知识点：{knowledge_point}
    - 难度等级：{difficulty}
    - 题型：{q_type}
    - 是否涉及运算：{calculation}
    - 认知层级（如：记忆 / 理解 / 应用 / 分析 / 评价 / 创造）：{cognitive_level}
    
    【出题要求】
    1. 题目必须以「目标知识点」为核心  
    2. 符合高中地理题风格
    3. 严格匹配给定的题型、难度、是否运算、认知层级
    4. 题干表述清晰，不引入无关信息。纯文本题，不要用到图片。
    5. 设问方式灵活，所问的可能是下一句或者选择一个陈述句（这两种情况最多），也可能是挖空，也可能是从下列选择或排序等
    6. 有概率用真实地理景观为例出题，比如“乞拉朋齐……据此请分析……的成因主要是”（不要一味参考这个例子）
    7. 只生成一道题，不要解释、不输出思路

    严格按照以下格式（包括换行也要一致）输出，不要含有其他内容如前后缀等，最前面不要有空格
    【输出格式】：
    【{difficulty}题 | {cognitive_level}题 | 运算题（如果涉及运算的话）】前两个xx分别填难度、认知层级
    【题目】
    （另起一行）题目内容
    （如是选择题）【选项】
    【参考答案】
    （另起一行）（如果是选择题，只输出选项一个字母）
    '''
)

In [73]:
#生成试题，不检索、不注入知识上下文
def rag_noncontext(llm,knowledge_point,difficulty,
        q_type,calculation,cognitive_level):

    response=llm.invoke(
        aprompt_noncontext.format(
            knowledge_point=knowledge_point,
            difficulty=difficulty,
            q_type=q_type,
            calculation=calculation,
            cognitive_level=cognitive_level
        )
    )

    return context,response.content

In [74]:
# 特征采样+试题生成 pipeline
def get_questions_noncontext(llm,block,stu_group,knowledge_point=None):
    # 抽取生成要求
    difficulty_p,calculation,cognitive_p=merge_feature(
        block=block,
        stu_group=stu_group)
    
    cur_difficulty,cur_calculation,cur_cognitive_level=choose_feature(
        difficulty_p,calculation,cognitive_p
    )
    # 抽取知识点
    all_kps=feature_map[block]['k_points']
    if not knowledge_point or knowledge_point not in allkps:
        knowledge_point=random.choices(all_kps,k=1)[0]
    # 生成
    context,answer=rag_noncontext(
        llm=llm,
        knowledge_point=knowledge_point,
        difficulty=cur_difficulty,
        q_type=q_type_list[0],
        calculation=cur_calculation,
        cognitive_level=cur_cognitive_level
    )

    features=[knowledge_point,cur_difficulty,q_type_list[0],
              cur_calculation,cur_cognitive_level,]

    return answer,features,context

In [75]:
# 解析llm输出->结构化字段 
def parse_noncontext(answer:str, idx:int, features, context):
    # print(answer)
    lines = [l.strip() for l in answer.strip().splitlines() if l.strip()]
    # print(lines)
    # 题干、选项、参考答案
    q_start = lines.index("【题目】") + 1
    o_start = lines.index("【选项】")
    a_start = lines.index("【参考答案】") + 1

    question = " ".join(lines[q_start:o_start])
    options = " ".join(lines[o_start + 1:a_start - 1])
    answer_text = lines[a_start]

    context = context.replace('\n', '')

    features.append(question)
    features.append(options)
    features.append(answer_text)
    features.append(context)
    return features

In [76]:
#批量生成、存表
llm = llm_qwen
n = 100
save_path = './output/comparison/raw/qwen-210-noncontext.txt'

count = 0  # 成功生成的题目数
attempt = 0  # 总尝试次数（防死循环）

with open(save_path, "w", encoding="utf-8") as f:
    while count < n and attempt < n * 3:  # 最多尝试 3n 次
        attempt += 1
        print(f'正在生成第{count+1}题（尝试第{attempt}次）：', end=' ')
        try:
            answer, features, context = get_questions_noncontext(
                llm=llm_qwen,
                block='流水地貌',
                stu_group=0,
                knowledge_point=None
            )
            features = parse_noncontext(answer, count, features, context)  # 这里用 count 做题号
            f.write(";".join(features) + "\n")
            f.flush()  # 立即写入，避免丢失
            count += 1
            print('成功')
        except Exception as e:
            print(f'【生成有误，重试】错误信息：{e}')

    print(f'生成完成，共成功 {count} 题，保存至 {save_path}')

正在生成第1题（尝试第1次）： 中等 不涉及 理解
成功
正在生成第2题（尝试第2次）： 中等 不涉及 理解
成功
正在生成第3题（尝试第3次）： 简单 不涉及 理解
成功
正在生成第4题（尝试第4次）： 中等 不涉及 理解
成功
正在生成第5题（尝试第5次）： 简单 不涉及 记忆
成功
正在生成第6题（尝试第6次）： 简单 不涉及 理解
成功
正在生成第7题（尝试第7次）： 中等 不涉及 理解
成功
正在生成第8题（尝试第8次）： 简单 不涉及 理解
成功
正在生成第9题（尝试第9次）： 中等 不涉及 理解
成功
正在生成第10题（尝试第10次）： 简单 不涉及 理解
成功
正在生成第11题（尝试第11次）： 简单 不涉及 理解
成功
正在生成第12题（尝试第12次）： 简单 不涉及 理解
成功
正在生成第13题（尝试第13次）： 简单 不涉及 理解
成功
正在生成第14题（尝试第14次）： 简单 不涉及 理解
成功
正在生成第15题（尝试第15次）： 简单 不涉及 分析
成功
正在生成第16题（尝试第16次）： 中等 不涉及 理解
成功
正在生成第17题（尝试第17次）： 简单 不涉及 理解
成功
正在生成第18题（尝试第18次）： 简单 不涉及 理解
成功
正在生成第19题（尝试第19次）： 困难 不涉及 理解
成功
正在生成第20题（尝试第20次）： 中等 不涉及 理解
成功
正在生成第21题（尝试第21次）： 简单 不涉及 理解
成功
正在生成第22题（尝试第22次）： 简单 不涉及 分析
成功
正在生成第23题（尝试第23次）： 简单 不涉及 理解
成功
正在生成第24题（尝试第24次）： 中等 不涉及 理解
成功
正在生成第25题（尝试第25次）： 简单 不涉及 理解
成功
正在生成第26题（尝试第26次）： 中等 不涉及 理解
成功
正在生成第27题（尝试第27次）： 简单 不涉及 分析
成功
正在生成第28题（尝试第28次）： 简单 不涉及 理解
成功
正在生成第29题（尝试第29次）： 简单 不涉及 理解
成功
正在生成第30题（尝试第30次）： 简单 不涉及 理解
成功
正在生成第31题（尝试第31次）： 中等 不涉及 分析
成功
正在生成第32题（尝试第32次）： 简单 不涉及 理解
成功
正在生成第33题（尝试第33次）： 中等 不涉及 理

In [77]:
#批量生成、存表
llm = llm_qwen
n = 100
save_path = './output/comparison/raw/qwen-220-noncontext.txt'

count = 0  # 成功生成的题目数
attempt = 0  # 总尝试次数（防死循环）

with open(save_path, "w", encoding="utf-8") as f:
    while count < n and attempt < n * 3:  # 最多尝试 3n 次
        attempt += 1
        print(f'正在生成第{count+1}题（尝试第{attempt}次）：', end=' ')
        try:
            answer, features, context = get_questions_noncontext(
                llm=llm_qwen,
                block='风成地貌',
                stu_group=0,
                knowledge_point=None
            )
            features = parse_noncontext(answer, count, features, context)  # 这里用 count 做题号
            f.write(";".join(features) + "\n")
            f.flush()  # 立即写入，避免丢失
            count += 1
            print('成功')
        except Exception as e:
            print(f'【生成有误，重试】错误信息：{e}')

    print(f'生成完成，共成功 {count} 题，保存至 {save_path}')

正在生成第1题（尝试第1次）： 中等 不涉及 分析
成功
正在生成第2题（尝试第2次）： 中等 不涉及 理解
成功
正在生成第3题（尝试第3次）： 简单 不涉及 理解
成功
正在生成第4题（尝试第4次）： 中等 不涉及 记忆
成功
正在生成第5题（尝试第5次）： 中等 不涉及 记忆
成功
正在生成第6题（尝试第6次）： 中等 不涉及 理解
成功
正在生成第7题（尝试第7次）： 中等 不涉及 理解
成功
正在生成第8题（尝试第8次）： 中等 不涉及 记忆
成功
正在生成第9题（尝试第9次）： 中等 不涉及 记忆
成功
正在生成第10题（尝试第10次）： 中等 不涉及 记忆
成功
正在生成第11题（尝试第11次）： 中等 不涉及 理解
成功
正在生成第12题（尝试第12次）： 简单 不涉及 记忆
成功
正在生成第13题（尝试第13次）： 中等 不涉及 记忆
成功
正在生成第14题（尝试第14次）： 中等 不涉及 理解
成功
正在生成第15题（尝试第15次）： 简单 不涉及 分析
成功
正在生成第16题（尝试第16次）： 简单 不涉及 分析
成功
正在生成第17题（尝试第17次）： 简单 不涉及 记忆
成功
正在生成第18题（尝试第18次）： 简单 不涉及 记忆
成功
正在生成第19题（尝试第19次）： 简单 不涉及 理解
成功
正在生成第20题（尝试第20次）： 中等 不涉及 记忆
成功
正在生成第21题（尝试第21次）： 简单 不涉及 分析
【生成有误，重试】错误信息：'【题目】' is not in list
正在生成第21题（尝试第22次）： 简单 不涉及 理解
成功
正在生成第22题（尝试第23次）： 简单 不涉及 理解
成功
正在生成第23题（尝试第24次）： 简单 不涉及 记忆
成功
正在生成第24题（尝试第25次）： 中等 不涉及 记忆
成功
正在生成第25题（尝试第26次）： 简单 不涉及 理解
成功
正在生成第26题（尝试第27次）： 中等 不涉及 理解
成功
正在生成第27题（尝试第28次）： 中等 不涉及 理解
成功
正在生成第28题（尝试第29次）： 简单 不涉及 记忆
成功
正在生成第29题（尝试第30次）： 中等 不涉及 理解
成功
正在生成第30题（尝试第31次）： 简单 不涉及 分析
成功
正在生成第31题（尝试第32次）： 中等 不涉及

In [78]:
#批量生成、存表
llm = llm_qwen
n = 100
save_path = './output/comparison/raw/qwen-230-noncontext.txt'

count = 0  # 成功生成的题目数
attempt = 0  # 总尝试次数（防死循环）

with open(save_path, "w", encoding="utf-8") as f:
    while count < n and attempt < n * 3:  # 最多尝试 3n 次
        attempt += 1
        print(f'正在生成第{count+1}题（尝试第{attempt}次）：', end=' ')
        try:
            answer, features, context = get_questions_noncontext(
                llm=llm_qwen,
                block='喀斯特、海岸和冰川地貌',
                stu_group=0,
                knowledge_point=None
            )
            features = parse_noncontext(answer, count, features, context)  # 这里用 count 做题号
            f.write(";".join(features) + "\n")
            f.flush()  # 立即写入，避免丢失
            count += 1
            print('成功')
        except Exception as e:
            print(f'【生成有误，重试】错误信息：{e}')

    print(f'生成完成，共成功 {count} 题，保存至 {save_path}')

正在生成第1题（尝试第1次）： 简单 不涉及 记忆
成功
正在生成第2题（尝试第2次）： 简单 不涉及 分析
成功
正在生成第3题（尝试第3次）： 简单 不涉及 分析
成功
正在生成第4题（尝试第4次）： 简单 不涉及 分析
成功
正在生成第5题（尝试第5次）： 简单 不涉及 分析
成功
正在生成第6题（尝试第6次）： 困难 不涉及 分析
成功
正在生成第7题（尝试第7次）： 简单 不涉及 分析
成功
正在生成第8题（尝试第8次）： 困难 不涉及 分析
成功
正在生成第9题（尝试第9次）： 简单 不涉及 分析
成功
正在生成第10题（尝试第10次）： 困难 不涉及 分析
成功
正在生成第11题（尝试第11次）： 困难 不涉及 分析
成功
正在生成第12题（尝试第12次）： 简单 不涉及 记忆
成功
正在生成第13题（尝试第13次）： 困难 不涉及 分析
【生成有误，重试】错误信息：'【题目】' is not in list
正在生成第13题（尝试第14次）： 简单 不涉及 记忆
成功
正在生成第14题（尝试第15次）： 简单 不涉及 分析
成功
正在生成第15题（尝试第16次）： 困难 不涉及 理解
成功
正在生成第16题（尝试第17次）： 困难 不涉及 记忆
成功
正在生成第17题（尝试第18次）： 困难 不涉及 记忆
成功
正在生成第18题（尝试第19次）： 中等 不涉及 记忆
成功
正在生成第19题（尝试第20次）： 简单 不涉及 分析
成功
正在生成第20题（尝试第21次）： 困难 不涉及 分析
成功
正在生成第21题（尝试第22次）： 简单 不涉及 分析
成功
正在生成第22题（尝试第23次）： 简单 不涉及 记忆
成功
正在生成第23题（尝试第24次）： 中等 不涉及 应用
成功
正在生成第24题（尝试第25次）： 困难 不涉及 分析
成功
正在生成第25题（尝试第26次）： 困难 不涉及 分析
成功
正在生成第26题（尝试第27次）： 困难 不涉及 分析
成功
正在生成第27题（尝试第28次）： 困难 不涉及 分析
成功
正在生成第28题（尝试第29次）： 困难 不涉及 分析
成功
正在生成第29题（尝试第30次）： 困难 不涉及 应用
成功
正在生成第30题（尝试第31次）： 简单 不涉及 理解
成功
正在生成第31题（尝试第32次）： 简单 不涉及

## 3.2 向量检索

In [34]:
from langchain_community.document_loaders import TextLoader
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings
import os
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [37]:
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com" 

In [40]:
# 构建向量库
md_path = './data/cleaned_湘教版地理必修一Chap2.md'

def build_vectorstore(md_path):
    loader = TextLoader(md_path, encoding='utf-8')
    docs = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(
        separators='\n\n',
        chunk_size=500,
        chunk_overlap=50
    )

    splits = text_splitter.split_documents(docs)
    
    embeddings=HuggingFaceEmbeddings(
        model_name='all-MiniLM-L6-v2'
    )
    vectorstore = FAISS.from_documents(splits, embeddings)
    return vectorstore


vs = build_vectorstore(md_path)
print('向量库构建完成')

向量库构建完成


In [41]:
# 向量检索函数
def retrieve_text(vectorstore, query, k=3):
    docs = vectorstore.similarity_search(query, k=k)
    return '\n\n'.join([d.page_content for d in docs])

# 测试检索
test_context = retrieve_text(vs, '牛轭湖')
print('检索结果：\n', test_context)

检索结果：
 （2）若对长江流域进行流水地貌考察，在上游、中下游及河口分别可观察到哪些地貌类型？说出判断理由。

（1）结合地图和照片，向同学介绍你游览过或通过其他途径了解到的岬角。

（2）在波浪不断侵蚀下，岬角海岸通常会出现哪些海蚀地貌？

（3）一些大洲的大陆以岬角为其四至。根据下列信息进行空间定位，判断甲、乙、丙三个大洲的名称。

甲大洲大陆最东端为哈丰角（ $5 1 ^ { \circ } \ 2 4 ^ { \prime }$ ′ E， $1 0 ^ { \circ } ~ 2 7 ^ { \prime } ~ \mathrm { ~ N ~ }$ ），最西端为佛得角（ $1 7 ^ { \circ } \ 3 3 ^ { \prime } \ \mathrm { ~ W ~ }$ ，$1 4 ^ { \circ } ~ 4 5 ^ { \prime } ~ \mathrm { ~ N ~ }$ ），最北端为本塞卡角（ $9 ^ { \circ } ~ 5 0 ^ { \prime } ~ \mathrm { ~ E ~ }$ ， $3 7 ^ { \circ } \ 2 1 ^ { \prime } \ \mathrm { ~ N ~ }$ ），最南端为厄加勒斯角（ $2 0 ^ { \circ } 0 2 ^ { \prime } \mathrm { ~ E ~ }$ ， $3 4 ^ { \circ } 5 1 ^ { \prime } \mathrm { ~ S ~ }$ ）。


In [42]:
prompt=PromptTemplate.from_template(
    '''
    你是一名高中地理资深命题教师，请根据以下任务说明进行出题。
    
    【已知条件】
    - 知识上下文：
    {context}
    - 目标知识点：{knowledge_point}
    - 难度等级：{difficulty}
    - 题型：{q_type}
    - 是否涉及运算：{calculation}
    - 认知层级（如：记忆 / 理解 / 应用 / 分析 / 评价 / 创造）：
    {cognitive_level}
    
    【出题要求】
    1. 题目必须以「目标知识点」为核心  
    2. 可合理使用上下文中的相关知识点作为支撑，但只能起辅助作用  
    3. 严格匹配给定的题型、难度、是否运算、认知层级
    4. 题干表述清晰，不引入无关信息。纯文本题，不要用到图片。
    5. 设问方式灵活，所问的可能是下一句或者选择一个陈述句（这两种情况最多），也可能是挖空，也可能是从下列选择或排序等
    5. 有概率用真实地理景观为例出题，但不要只用上下文中的，可以用其他的，比如“乞拉朋齐……据此请分析……的成因主要是”
    5. 只生成一道题，不要解释、不输出思路
    6. 要符合高中地理考题风格

    严格按照以下格式（包括换行也要一致）输出，不要含有其他内容如前后缀等，最前面不要有空格
    【输出格式】：
    【{difficulty}题 | {cognitive_level}题 | 运算题（如果涉及运算的话）】前两个xx分别填难度、认知层级
    【题目】
    （另起一行）题目内容
    （如是选择题）【选项】
    【参考答案】
    （另起一行）（如果是选择题，只输出选项一个字母）
    '''
)

In [43]:
# 生成pipeline（用户自己替换prompt模板）
def rag_with_text(vectorstore, knowledge_point, difficulty,
        q_type,calculation,cognitive_level):
    # 向量检索获取上下文
    context = retrieve_text(vectorstore, knowledge_point)
    # 生成（用户替换这里的prompt模板）
    response = llm.invoke(
        prompt.format(
            context=context,
            knowledge_point=knowledge_point,
            difficulty=difficulty,
            q_type=q_type,
            calculation=calculation,
            cognitive_level=cognitive_level
        )
    )
    return context, response.content

In [44]:
# 特征采样+生成pipeline
def get_questions_text(block, stu_group, knowledge_point=None):
    # 抽取生成要求
    difficulty_p, calculation, cognitive_p = merge_feature(
        block=block,
        stu_group=stu_group)
    
    cur_difficulty, cur_calculation, cur_cognitive_level = choose_feature(
        difficulty_p, calculation, cognitive_p
    )
    # 抽取知识点
    all_kps = feature_map[block]['k_points']
    if not knowledge_point or knowledge_point not in all_kps:
        knowledge_point = random.choices(all_kps, k=1)[0]
    # 生成
    context, answer = rag_with_text(
        vs,  # 使用全局向量库
        knowledge_point=knowledge_point,
        difficulty=cur_difficulty,
        q_type=q_type_list[0],
        calculation=cur_calculation,
        cognitive_level=cur_cognitive_level
    )

    features=[knowledge_point, cur_difficulty, q_type_list[0],
              cur_calculation, cur_cognitive_level,]

    return answer, features, context

In [64]:
# 批量生成（示例）
def generate_batch_nontext(llm,n,block,stu):
    if llm=='ds':
        llm=llm_ds
        save_path = f'./output/ds-2{block}{stu}-text-rag.txt'
    elif llm=='qwen':
        llm=llm_qwen
        save_path = f'./output/qwen-2{block}{stu}-text-rag.txt'
        
    print('will save to:',save_path)
    
    count = 0
    attempt = 0
    blocks=['流水地貌','风成地貌','喀斯特、海岸和冰川地貌']
    
    with open(save_path, "w", encoding="utf-8") as f:
        while count < n and attempt < n * 3:
            attempt += 1
            print(f'正在生成第{count+1}题（尝试第{attempt}次）：', end=' ')
            try:
                answer, features, context = get_questions_text(
                    block=blocks[block-1],
                    stu_group=stu,
                    knowledge_point=None
                )
                features = parse(answer, count, features, context)
                f.write(";".join(features) + "\n")
                f.flush()
                count += 1
                print('成功')
            except Exception as e:
                print(f'【生成有误，重试】错误信息：{e}')
    
    print(f'生成完成，共成功 {count} 题，保存至 {save_path}')

In [52]:
generate_batch_nontext(llm_qwen,n=100,block=2,stu=0)

will save to: ./output/ds-220-text-rag.txt
正在生成第1题（尝试第1次）： 简单 不涉及 记忆
成功
正在生成第2题（尝试第2次）： 简单 不涉及 应用
成功
正在生成第3题（尝试第3次）： 中等 不涉及 理解
成功
正在生成第4题（尝试第4次）： 中等 不涉及 分析
成功
正在生成第5题（尝试第5次）： 中等 不涉及 理解
成功
正在生成第6题（尝试第6次）： 简单 不涉及 记忆
成功
正在生成第7题（尝试第7次）： 中等 不涉及 记忆
成功
正在生成第8题（尝试第8次）： 简单 不涉及 理解
成功
正在生成第9题（尝试第9次）： 简单 不涉及 分析
成功
正在生成第10题（尝试第10次）： 中等 不涉及 分析
成功
正在生成第11题（尝试第11次）： 中等 不涉及 分析
成功
正在生成第12题（尝试第12次）： 简单 不涉及 理解
成功
正在生成第13题（尝试第13次）： 中等 不涉及 记忆
成功
正在生成第14题（尝试第14次）： 简单 不涉及 理解
成功
正在生成第15题（尝试第15次）： 中等 不涉及 记忆
成功
正在生成第16题（尝试第16次）： 中等 不涉及 理解
成功
正在生成第17题（尝试第17次）： 中等 不涉及 分析
成功
正在生成第18题（尝试第18次）： 简单 不涉及 分析
成功
正在生成第19题（尝试第19次）： 中等 不涉及 分析
成功
正在生成第20题（尝试第20次）： 简单 不涉及 理解
成功
正在生成第21题（尝试第21次）： 中等 不涉及 记忆
成功
正在生成第22题（尝试第22次）： 简单 不涉及 分析
成功
正在生成第23题（尝试第23次）： 中等 不涉及 理解
成功
正在生成第24题（尝试第24次）： 简单 不涉及 记忆
成功
正在生成第25题（尝试第25次）： 中等 不涉及 记忆
成功
正在生成第26题（尝试第26次）： 简单 不涉及 理解
成功
正在生成第27题（尝试第27次）： 中等 不涉及 理解
成功
正在生成第28题（尝试第28次）： 中等 不涉及 记忆
成功
正在生成第29题（尝试第29次）： 中等 不涉及 记忆
成功
正在生成第30题（尝试第30次）： 简单 不涉及 分析
成功
正在生成第31题（尝试第31次）： 简单 不涉及 记忆
成功
正在生成第32题（尝试第32

In [53]:
generate_batch_nontext(llm_ds,n=100,block=3,stu=0)

will save to: ./output/ds-230-text-rag.txt
正在生成第1题（尝试第1次）： 简单 不涉及 记忆
成功
正在生成第2题（尝试第2次）： 简单 不涉及 分析
成功
正在生成第3题（尝试第3次）： 简单 不涉及 分析
成功
正在生成第4题（尝试第4次）： 简单 不涉及 分析
成功
正在生成第5题（尝试第5次）： 困难 不涉及 分析
成功
正在生成第6题（尝试第6次）： 困难 不涉及 分析
成功
正在生成第7题（尝试第7次）： 简单 不涉及 分析
成功
正在生成第8题（尝试第8次）： 简单 不涉及 分析
成功
正在生成第9题（尝试第9次）： 简单 不涉及 分析
成功
正在生成第10题（尝试第10次）： 困难 不涉及 分析
成功
正在生成第11题（尝试第11次）： 简单 不涉及 记忆
成功
正在生成第12题（尝试第12次）： 困难 不涉及 记忆
成功
正在生成第13题（尝试第13次）： 简单 不涉及 记忆
成功
正在生成第14题（尝试第14次）： 简单 不涉及 分析
成功
正在生成第15题（尝试第15次）： 困难 不涉及 分析
成功
正在生成第16题（尝试第16次）： 简单 不涉及 分析
成功
正在生成第17题（尝试第17次）： 困难 不涉及 分析
成功
正在生成第18题（尝试第18次）： 简单 不涉及 分析
成功
正在生成第19题（尝试第19次）： 简单 不涉及 记忆
成功
正在生成第20题（尝试第20次）： 简单 不涉及 理解
成功
正在生成第21题（尝试第21次）： 简单 不涉及 分析
成功
正在生成第22题（尝试第22次）： 简单 不涉及 分析
成功
正在生成第23题（尝试第23次）： 简单 不涉及 分析
成功
正在生成第24题（尝试第24次）： 困难 不涉及 分析
成功
正在生成第25题（尝试第25次）： 简单 不涉及 分析
成功
正在生成第26题（尝试第26次）： 简单 不涉及 分析
成功
正在生成第27题（尝试第27次）： 简单 不涉及 应用
成功
正在生成第28题（尝试第28次）： 简单 不涉及 记忆
成功
正在生成第29题（尝试第29次）： 简单 不涉及 分析
成功
正在生成第30题（尝试第30次）： 困难 不涉及 理解
成功
正在生成第31题（尝试第31次）： 中等 不涉及 记忆
成功
正在生成第32题（尝试第32

In [65]:
generate_batch_nontext('qwen',n=100,block=1,stu=0)

will save to: ./output/qwen-210-text-rag.txt
正在生成第1题（尝试第1次）： 简单 不涉及 理解
成功
正在生成第2题（尝试第2次）： 中等 不涉及 理解
成功
正在生成第3题（尝试第3次）： 简单 不涉及 分析
成功
正在生成第4题（尝试第4次）： 简单 不涉及 分析
成功
正在生成第5题（尝试第5次）： 简单 不涉及 分析
成功
正在生成第6题（尝试第6次）： 简单 不涉及 理解
成功
正在生成第7题（尝试第7次）： 简单 不涉及 理解
成功
正在生成第8题（尝试第8次）： 简单 不涉及 理解
成功
正在生成第9题（尝试第9次）： 简单 不涉及 记忆
成功
正在生成第10题（尝试第10次）： 中等 不涉及 分析
成功
正在生成第11题（尝试第11次）： 简单 不涉及 理解
成功
正在生成第12题（尝试第12次）： 简单 不涉及 分析
成功
正在生成第13题（尝试第13次）： 简单 不涉及 理解
成功
正在生成第14题（尝试第14次）： 简单 不涉及 理解
成功
正在生成第15题（尝试第15次）： 简单 不涉及 理解
成功
正在生成第16题（尝试第16次）： 简单 不涉及 理解
成功
正在生成第17题（尝试第17次）： 中等 不涉及 分析
成功
正在生成第18题（尝试第18次）： 中等 不涉及 理解
成功
正在生成第19题（尝试第19次）： 简单 不涉及 理解
成功
正在生成第20题（尝试第20次）： 简单 不涉及 分析
成功
正在生成第21题（尝试第21次）： 简单 不涉及 理解
成功
正在生成第22题（尝试第22次）： 简单 不涉及 理解
成功
正在生成第23题（尝试第23次）： 简单 不涉及 理解
成功
正在生成第24题（尝试第24次）： 简单 不涉及 分析
成功
正在生成第25题（尝试第25次）： 简单 不涉及 分析
成功
正在生成第26题（尝试第26次）： 简单 不涉及 理解
成功
正在生成第27题（尝试第27次）： 简单 不涉及 分析
成功
正在生成第28题（尝试第28次）： 简单 不涉及 理解
成功
正在生成第29题（尝试第29次）： 中等 不涉及 理解
成功
正在生成第30题（尝试第30次）： 简单 不涉及 理解
成功
正在生成第31题（尝试第31次）： 简单 不涉及 理解
成功
正在生成第32题（尝试第

In [67]:
generate_batch_nontext('qwen',n=100,block=2,stu=0)

will save to: ./output/qwen-220-text-rag.txt
正在生成第1题（尝试第1次）： 中等 不涉及 分析
成功
正在生成第2题（尝试第2次）： 中等 不涉及 应用
成功
正在生成第3题（尝试第3次）： 中等 不涉及 分析
成功
正在生成第4题（尝试第4次）： 中等 不涉及 理解
成功
正在生成第5题（尝试第5次）： 简单 不涉及 记忆
成功
正在生成第6题（尝试第6次）： 中等 不涉及 应用
成功
正在生成第7题（尝试第7次）： 中等 不涉及 记忆
成功
正在生成第8题（尝试第8次）： 中等 不涉及 记忆
成功
正在生成第9题（尝试第9次）： 中等 不涉及 理解
成功
正在生成第10题（尝试第10次）： 中等 不涉及 记忆
成功
正在生成第11题（尝试第11次）： 简单 不涉及 分析
成功
正在生成第12题（尝试第12次）： 中等 不涉及 分析
成功
正在生成第13题（尝试第13次）： 中等 不涉及 记忆
成功
正在生成第14题（尝试第14次）： 简单 不涉及 理解
成功
正在生成第15题（尝试第15次）： 中等 不涉及 记忆
成功
正在生成第16题（尝试第16次）： 中等 不涉及 记忆
成功
正在生成第17题（尝试第17次）： 简单 不涉及 理解
成功
正在生成第18题（尝试第18次）： 中等 不涉及 记忆
成功
正在生成第19题（尝试第19次）： 中等 不涉及 分析
成功
正在生成第20题（尝试第20次）： 中等 不涉及 理解
成功
正在生成第21题（尝试第21次）： 中等 不涉及 分析
成功
正在生成第22题（尝试第22次）： 简单 不涉及 理解
成功
正在生成第23题（尝试第23次）： 中等 不涉及 理解
成功
正在生成第24题（尝试第24次）： 中等 不涉及 记忆
成功
正在生成第25题（尝试第25次）： 简单 不涉及 分析
成功
正在生成第26题（尝试第26次）： 简单 不涉及 分析
成功
正在生成第27题（尝试第27次）： 中等 不涉及 理解
成功
正在生成第28题（尝试第28次）： 简单 不涉及 理解
成功
正在生成第29题（尝试第29次）： 中等 不涉及 理解
成功
正在生成第30题（尝试第30次）： 中等 不涉及 分析
成功
正在生成第31题（尝试第31次）： 中等 不涉及 记忆
成功
正在生成第32题（尝试第

In [68]:
generate_batch_nontext('qwen',n=100,block=3,stu=0)

will save to: ./output/qwen-230-text-rag.txt
正在生成第1题（尝试第1次）： 困难 不涉及 理解
成功
正在生成第2题（尝试第2次）： 困难 不涉及 记忆
成功
正在生成第3题（尝试第3次）： 困难 不涉及 理解
成功
正在生成第4题（尝试第4次）： 困难 不涉及 分析
成功
正在生成第5题（尝试第5次）： 简单 不涉及 分析
成功
正在生成第6题（尝试第6次）： 困难 不涉及 记忆
成功
正在生成第7题（尝试第7次）： 困难 不涉及 记忆
成功
正在生成第8题（尝试第8次）： 困难 不涉及 记忆
成功
正在生成第9题（尝试第9次）： 困难 不涉及 记忆
成功
正在生成第10题（尝试第10次）： 困难 不涉及 分析
成功
正在生成第11题（尝试第11次）： 困难 不涉及 应用
成功
正在生成第12题（尝试第12次）： 简单 不涉及 分析
成功
正在生成第13题（尝试第13次）： 简单 不涉及 记忆
成功
正在生成第14题（尝试第14次）： 简单 不涉及 理解
成功
正在生成第15题（尝试第15次）： 困难 不涉及 分析
成功
正在生成第16题（尝试第16次）： 简单 不涉及 理解
成功
正在生成第17题（尝试第17次）： 简单 不涉及 记忆
成功
正在生成第18题（尝试第18次）： 简单 不涉及 分析
成功
正在生成第19题（尝试第19次）： 简单 不涉及 分析
成功
正在生成第20题（尝试第20次）： 困难 不涉及 记忆
成功
正在生成第21题（尝试第21次）： 困难 不涉及 分析
成功
正在生成第22题（尝试第22次）： 困难 不涉及 应用
成功
正在生成第23题（尝试第23次）： 困难 不涉及 分析
成功
正在生成第24题（尝试第24次）： 简单 不涉及 分析
成功
正在生成第25题（尝试第25次）： 困难 不涉及 记忆
成功
正在生成第26题（尝试第26次）： 困难 不涉及 分析
成功
正在生成第27题（尝试第27次）： 简单 不涉及 分析
成功
正在生成第28题（尝试第28次）： 简单 不涉及 分析
成功
正在生成第29题（尝试第29次）： 简单 不涉及 分析
成功
正在生成第30题（尝试第30次）： 困难 不涉及 分析
成功
正在生成第31题（尝试第31次）： 困难 不涉及 记忆
成功
正在生成第32题（尝试第

## 3.3 图检索（前面写过了）